# 03 — Preprocessing Pipeline Demo (Paper Fig. 2)

Demonstrates the 6-step preprocessing pipeline:
1. Load raw DICOM
2. YOLO breast ROI detection
3. Square crop around ROI
4. Threshold (suppress background noise, value=20)
5. Resize to 512×512
6. Save as JPEG (quality=100)

Note: CLAHE is **not** applied offline. It is applied online during training/inference.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from multiview_davit.config import load_config
from multiview_davit.preprocessing import read_img, load_yolov5, keep_breast_roi, to_u8

cfg = load_config('configs/preprocess.yaml')

# Set paths
SAMPLE_DICOM = 'data/raw/cbis_ddsm/sample.dcm'  # replace with actual path
YOLO_WEIGHTS = cfg.yolo.weights
YOLOV5_PATH  = os.environ.get('YOLOV5_PATH', '')  # set YOLOV5_PATH env var

In [ ]:
# Step 1: Load raw DICOM
raw = read_img(SAMPLE_DICOM)
print(f'Raw shape: {raw.shape}, dtype: {raw.dtype}')

fig, axes = plt.subplots(1, 6, figsize=(24, 4))
axes[0].imshow(to_u8(raw), cmap='gray')
axes[0].set_title('1. Raw DICOM')

In [ ]:
# Steps 2-3: YOLO detection + square crop
runner = load_yolov5(YOLO_WEIGHTS, yolov5_path=YOLOV5_PATH)
crop = keep_breast_roi(SAMPLE_DICOM, model=runner)
axes[1].imshow(to_u8(crop), cmap='gray')
axes[1].set_title('2-3. ROI Crop')

In [ ]:
# Step 4: Threshold
img_u8 = to_u8(crop if crop.ndim == 2 else cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY))
thresholded = cv2.threshold(img_u8, cfg.threshold_value, 255, cv2.THRESH_TOZERO)[1]
axes[2].imshow(thresholded, cmap='gray')
axes[2].set_title('4. Threshold')

# Step 5: Resize
resized = cv2.resize(thresholded, (cfg.out_size, cfg.out_size), interpolation=cv2.INTER_AREA)
axes[3].imshow(resized, cmap='gray')
axes[3].set_title(f'5. Resize {cfg.out_size}×{cfg.out_size}')

for ax in axes[4:]:
    ax.axis('off')
plt.tight_layout()
plt.show()